# Challenge 1: Weather Chatbot with the Google ADK

A friendly chatbot. You ask *"Hey, what is the weather like in Denver?"* and it answers with the
current conditions.

**How it works**

1. `get_lat_lon` - turns the place name into coordinates with the **Google Maps Geocoding API**
2. `get_current_weather` - gets the conditions for those coordinates from the **National Weather Service API**
3. An ADK `LlmAgent` decides when to call each tool and writes the friendly reply

The same agent runs on **Gemini** and on a **third-party model** (GPT via LiteLLM), and a short loop
tests it against several U.S. cities.

## 1. Install and configure

In [1]:
%pip install -q --upgrade google-adk "litellm>=1.84" requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [11]:
import os
import google.auth

# Use the lab project's own credentials for Gemini on Vertex AI (Agent Platform).
_, _project = google.auth.default()
PROJECT_ID = _project or "qwiklabs-gcp-03-9b118d33ce6b"   # fallback: paste your project ID here

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

MODEL_GEMINI = "gemini-3.8-flash"
print("Project:", PROJECT_ID, "| Model:", MODEL_GEMINI)

Project: qwiklabs-gcp-03-9b118d33ce6b | Model: gemini-3.8-flash


## 2. The two tools

Plain Python functions. ADK reads the type hints and docstrings to learn how to call them.

In [12]:
from typing import Any, Dict
from urllib.parse import quote

import requests
import google.auth.transport.requests


def get_lat_lon(place: str) -> Dict[str, Any]:
    """Convert a place name to latitude/longitude using the Google Maps Geocoding API.

    Args:
        place (str): A city, address or landmark, e.g. "Denver, CO".

    Returns:
        Dict[str, Any]: ``{"status": "success", "place": str, "lat": float, "lon": float}``
        or ``{"status": "error", "error_message": str}``.
    """
    # 1) Google Maps Geocoding API (v4). Uses an API key if you have one, otherwise the
    #    lab project's own credentials (student accounts cannot create API keys).
    try:
        if os.environ.get("GOOGLE_MAPS_API_KEY"):
            headers = {"X-Goog-Api-Key": os.environ["GOOGLE_MAPS_API_KEY"]}
        else:
            creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
            creds.refresh(google.auth.transport.requests.Request())
            headers = {"Authorization": f"Bearer {creds.token}", "X-Goog-User-Project": PROJECT_ID}
        resp = requests.get(f"https://geocode.googleapis.com/v4/geocode/address/{quote(place)}", headers=headers, timeout=15)
        results = resp.json().get("results", [])
        if results:
            best = results[0]
            return {"status": "success", "source": "Google Maps", "place": best["formattedAddress"],
                    "lat": best["location"]["latitude"], "lon": best["location"]["longitude"]}
        print(f"   (Google Maps geocoding unavailable: HTTP {resp.status_code} {resp.text[:120]!r} - using fallback)")
    except Exception as exc:
        print(f"   (Google Maps geocoding unavailable: {str(exc)[:120]} - using fallback)")

    # 2) Fallback: Open-Meteo's free geocoder (no key). Looks up the city name, U.S. only.
    try:
        city = place.split(",")[0].strip()
        resp = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                            params={"name": city, "count": 1, "countryCode": "US"}, timeout=15)
        results = resp.json().get("results", [])
        if not results:
            return {"status": "error", "error_message": f"Could not find a U.S. place called '{place}'."}
        best = results[0]
        return {"status": "success", "source": "Open-Meteo", "place": f"{best['name']}, {best.get('admin1', '')}, USA",
                "lat": best["latitude"], "lon": best["longitude"]}
    except Exception as exc:  # keep the bot alive; report the problem to the model instead
        return {"status": "error", "error_message": f"Geocoding failed: {exc}"}


def get_current_weather(lat: float, lon: float) -> Dict[str, Any]:
    """Get the current weather conditions for a U.S. location from the National Weather Service API.

    Args:
        lat (float): Latitude in decimal degrees, e.g. 39.7392.
        lon (float): Longitude in decimal degrees, e.g. -104.9903.

    Returns:
        Dict[str, Any]: ``{"status": "success", "location": str, "period": str, "temperature": str,
        "wind": str, "conditions": str, "details": str}`` or ``{"status": "error", "error_message": str}``.
        Only United States locations are supported.
    """
    headers = {"User-Agent": "(adk-weather-workshop, student@example.com)"}  # NWS requires a User-Agent
    try:
        points = requests.get(f"https://api.weather.gov/points/{lat:.4f},{lon:.4f}", headers=headers, timeout=15)
        if points.status_code != 200:
            return {"status": "error", "error_message": "The National Weather Service only covers U.S. locations."}
        props = points.json()["properties"]

        forecast = requests.get(props["forecast"], headers=headers, timeout=15).json()
        now = forecast["properties"]["periods"][0]
        where = props["relativeLocation"]["properties"]
        return {
            "status": "success",
            "location": f"{where['city']}, {where['state']}",
            "period": now["name"],
            "temperature": f"{now['temperature']}°{now['temperatureUnit']}",
            "wind": f"{now['windSpeed']} {now['windDirection']}",
            "conditions": now["shortForecast"],
            "details": now["detailedForecast"],
        }
    except Exception as exc:
        return {"status": "error", "error_message": f"Weather lookup failed: {exc}"}


# Quick check that both tools work before giving them to the model
coords = get_lat_lon("Denver, CO")
print(coords)
print(get_current_weather(coords["lat"], coords["lon"]))

{'status': 'success', 'source': 'Google Maps', 'place': 'Denver, CO, USA', 'lat': 39.739235799999996, 'lon': -104.99025100000001}
{'status': 'success', 'location': 'Denver, CO', 'period': 'Today', 'temperature': '65°F', 'wind': '8 mph NE', 'conditions': 'Areas Of Drizzle then Mostly Cloudy', 'details': 'Areas of drizzle and patchy fog before noon. Mostly cloudy. High near 65, with temperatures falling to around 62 in the afternoon. Northeast wind around 8 mph. Chance of precipitation is 30%.'}


## 3. The agent

In [14]:
from google.adk.agents import LlmAgent

INSTRUCTIONS = """You are a friendly weather chatbot for U.S. locations.

When someone asks about the weather somewhere:
1. Call get_lat_lon with the place name to get its coordinates.
2. Call get_current_weather with those coordinates.
3. Reply in a warm, conversational tone (2-4 sentences) with the temperature, conditions and wind.
   Add a practical tip (umbrella, sunscreen, jacket...) when it fits.

If a tool returns an error, explain it simply, e.g. that you only cover U.S. locations.
Only answer weather questions."""


def make_weather_agent(model, name: str = "weather_bot") -> LlmAgent:
    """Build the chatbot on any model: a Gemini model name or a LiteLlm instance."""
    return LlmAgent(
        name=name,
        model=model,
        description="Friendly chatbot that reports current U.S. weather conditions.",
        instruction=INSTRUCTIONS,
        tools=[get_lat_lon, get_current_weather],
    )


weather_bot = make_weather_agent(MODEL_GEMINI)

## 4. Chat with it

In [15]:
import uuid

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP = "weather_app"
session_service = InMemorySessionService()


async def chat(agent: LlmAgent, message: str) -> str:
    """Send one message to the agent, show which tools it used, and print its reply."""
    session_id = uuid.uuid4().hex
    await session_service.create_session(app_name=APP, user_id="user", session_id=session_id)
    runner = Runner(agent=agent, app_name=APP, session_service=session_service)

    print(f"You: {message}")
    reply = ""
    async for event in runner.run_async(
        user_id="user", session_id=session_id, new_message=types.Content(role="user", parts=[types.Part(text=message)])
    ):
        for call in event.get_function_calls():
            print(f"   [tool] {call.name}({call.args})")
        if event.is_final_response() and event.content and event.content.parts:
            reply = "".join(part.text or "" for part in event.content.parts)
    print(f"Bot: {reply}\n")
    return reply


await chat(weather_bot, "Hey, what is the weather like in Denver?")

You: Hey, what is the weather like in Denver?
   [tool] get_lat_lon({'place': 'Denver, CO'})
   [tool] get_current_weather({'lon': -104.990251, 'lat': 39.7392358})
Bot: Right now in Denver, it's around 65°F with mostly cloudy skies and areas of drizzle. You'll also feel a gentle northeast breeze blowing at about 8 mph. It's a good idea to grab a light jacket and keep an umbrella handy in case you run into any damp spots!



"Right now in Denver, it's around 65°F with mostly cloudy skies and areas of drizzle. You'll also feel a gentle northeast breeze blowing at about 8 mph. It's a good idea to grab a light jacket and keep an umbrella handy in case you run into any damp spots!"

## 5. Test: several U.S. cities

In [18]:
import asyncio

TEST_CITIES = ["Miami, FL", "Seattle, WA", "Chicago, IL", "Phoenix, AZ", "Honolulu, HI"]


async def chat_with_retry(agent: LlmAgent, message: str, attempts: int = 3) -> str:
    """Retry a chat if the model API rate-limits or hiccups (lab projects have small quotas)."""
    for attempt in range(1, attempts + 1):
        try:
            return await chat(agent, message)
        except Exception as exc:
            print(f"   (attempt {attempt} failed: {str(exc)[:120]} - retrying in 20s)")
            await asyncio.sleep(20)
    raise RuntimeError(f"Gave up after {attempts} attempts: {message}")


for city in TEST_CITIES:
    reply = await chat_with_retry(weather_bot, f"What's the weather like in {city} right now?")
    assert reply, f"No reply for {city}"
    await asyncio.sleep(3)  # go easy on the quota

print(f"All {len(TEST_CITIES)} cities answered.")

You: What's the weather like in Miami, FL right now?
   [tool] get_lat_lon({'place': 'Miami, FL'})
   [tool] get_current_weather({'lat': 25.7616798, 'lon': -80.1917902})
Bot: It's currently a bright and sunny day in Miami, FL, with a warm temperature around 87°F! There's a nice northeast breeze blowing at 12 to 15 mph to help keep things comfortable. If you're heading outside, make sure to pack some sunscreen and sunglasses to protect yourself from the sunshine!

You: What's the weather like in Seattle, WA right now?
   [tool] get_lat_lon({'place': 'Seattle, WA'})
   [tool] get_current_weather({'lon': -122.3328481, 'lat': 47.6061389})
Bot: It's currently mostly cloudy in Seattle, WA, with a temperature around 60°F and a gentle breeze from the south-southeast at about 3 mph. It's relatively mild out, but with the cloud cover, you might want to bring along a light jacket or sweater if you're heading outside!

You: What's the weather like in Chicago, IL right now?
   [tool] get_lat_lon({'

## 6. Same bot on a third-party model

ADK's `LiteLlm` wrapper swaps the model and nothing else. Paste an OpenAI key when prompted
(or press Enter to skip). For Claude use `LiteLlm(model="anthropic/claude-sonnet-5-5")` with an
`ANTHROPIC_API_KEY` instead.

In [19]:
from getpass import getpass

from google.adk.models.lite_llm import LiteLlm

os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key (Enter to skip): ").strip()

if os.environ["OPENAI_API_KEY"]:
    gpt_bot = make_weather_agent(LiteLlm(model="openai/gpt-4o"), name="weather_bot_gpt")
    for city in TEST_CITIES[:3]:
        await chat(gpt_bot, f"What's the weather like in {city} right now?")
else:
    print("Skipped: no OpenAI key provided.")

OpenAI API key (Enter to skip): ··········
Skipped: no OpenAI key provided.
